# `CUSUM` runtime: processing 10k observations of dimension p=1000

A minimal benchmark of how long the `GridDetector` + `CUSUM` score takes to
process 10,000 observations of dimension p=1000, one sample at a time.

The first call to a Numba-compiled score function pays a one-time compilation
cost, so we **warm up** the kernels before timing. We then time the full
10k-sample loop over several independent repetitions and report the mean.

In [ ]:
import time

import numpy as np

from gridcp import GridDetector
from gridcp.scores import CUSUM

N = 10_000  # number of observations per run
N_REPEATS = 10  # number of timed repetitions to average over
SEED = 0
p = 1000

In [ ]:
# A high threshold means the detector never alarms, so every run processes
# all N observations (we are timing throughput, not detection).
score = CUSUM(n_features=p)
detector = GridDetector(score=score, threshold=1e18)


def process_stream(data):
    """Feed an array through the detector one sample at a time."""
    state = detector.init_state()
    for y in data:
        state, output = detector.update(state, y)
    return output

In [ ]:
# Warm-up: trigger Numba compilation (and disk cache) before timing.
rng = np.random.default_rng(SEED)
_ = process_stream(rng.standard_normal((N, p)))
print("Warm-up done (kernels compiled).")

In [ ]:
# Timed runs: fresh data and a fresh state each repetition.
times = []
for r in range(N_REPEATS):
    data = rng.standard_normal((N, p))
    t0 = time.perf_counter()
    process_stream(data)
    t1 = time.perf_counter()
    times.append(t1 - t0)

times = np.array(times)
print(f"Per-run times (s): {np.round(times, 3)}")
print(f"Mean over {N_REPEATS} runs: {times.mean():.3f} s  (std {times.std():.3f} s)")
print(f"Throughput: {N / times.mean():,.0f} observations/second")